In [1]:
import os
from dotenv import load_dotenv
from azure.ai.ml import MLClient, Input, Output
from azure.ai.ml.entities import Environment, AmlCompute
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import command
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.identity import InteractiveBrowserCredential

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────

load_dotenv("../.env")
# El ../ sube un nivel desde notebooks/ a la raíz del proyecto

subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID")
resource_group  = "rg-credit-risk-ml"
workspace_name  = "ws-credit-risk-ml"

credential = InteractiveBrowserCredential(
    tenant_id="adcc6d74-c336-483f-b3e0-508ef38f11d8"
)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

print(f"✅ Subscription: {subscription_id[:8]}...")
print(f"✅ Workspace: {ml_client.workspace_name}")
print(f"✅ Listo para configurar pipeline")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ Subscription: 7a434baa...
✅ Workspace: ws-credit-risk-ml
✅ Listo para configurar pipeline


In [2]:
# ─────────────────────────────────────────────
# CREAR COMPUTE CLUSTER
# ─────────────────────────────────────────────

compute_name = "cpu-cluster-cr"
# Nombre del cluster de cómputo

compute_config = AmlCompute(
    name=compute_name,
    type="amlcompute",
    # amlcompute: cluster administrado por Azure ML
    # escala automáticamente según la demanda

    size="Standard_DS2_v2",
    # VM más económica disponible para pipelines
    # 2 vCPUs, 7GB RAM — suficiente para reentrenamiento

    min_instances=0,
    # 0 instancias mínimas — el cluster se apaga
    # completamente cuando no hay jobs corriendo
    # CRÍTICO para no consumir crédito innecesariamente

    max_instances=1,
    # Máximo 1 instancia — suficiente para este pipeline

    idle_time_before_scale_down=120
    # Segundos de inactividad antes de apagar la instancia
    # 120 segundos = 2 minutos — apagado rápido para ahorrar crédito
)

ml_client.compute.begin_create_or_update(compute_config).result()
print(f"✅ Compute cluster '{compute_name}' listo")

C:\Users\Marin\.conda\envs\credit-risk\Lib\site-packages\msal\oauth2cli\oauth2.py:408: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


✅ Compute cluster 'cpu-cluster-cr' listo


In [3]:
# ─────────────────────────────────────────────
# DEFINIR ENTORNO DEL PIPELINE
# ─────────────────────────────────────────────

pipeline_env = Environment(
    name="credit-risk-pipeline-env",
    description="Entorno para pipeline de reentrenamiento de scoring crediticio",
    conda_file="../src/endpoint/environment.yml",
    # Reutilizamos el mismo environment.yml que creamos
    # para el endpoint — tiene todas las dependencias necesarias
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
    # Imagen base de Docker proporcionada por Microsoft
)

ml_client.environments.create_or_update(pipeline_env)
print(f"✅ Entorno '{pipeline_env.name}' registrado en Azure ML")

✅ Entorno 'credit-risk-pipeline-env' registrado en Azure ML


In [22]:
# ─────────────────────────────────────────────
# DEFINIR COMPONENTES DEL PIPELINE
# ─────────────────────────────────────────────

# Paso 1 — Preparar datos
prep_data_component = command(
    name="prep_data",
    display_name="Preparar datos",
    description="Limpieza y preparación del dataset para reentrenamiento",
    inputs={
        "input_data": Input(type=AssetTypes.URI_FOLDER)
        # URI_FOLDER: apunta a una carpeta en Azure Storage
    },
    outputs={
        "output_data": Output(type=AssetTypes.URI_FOLDER)
    },
    code="../src/pipeline",
    # Carpeta con los scripts del pipeline
    command="python prep_data.py --input_data ${{inputs.input_data}} --output_data ${{outputs.output_data}}",
    # Azure ML sustituye ${{inputs.x}} y ${{outputs.x}}
    # con las rutas reales en tiempo de ejecución
environment=f"credit-risk-pipeline-env-v5@latest",
    compute=compute_name
)

# Paso 2 — Entrenar modelo
train_component = command(
    name="train",
    display_name="Entrenar modelo",
    description="Reentrenamiento de LightGBM con datos preparados",
    inputs={
        "input_data": Input(type=AssetTypes.URI_FOLDER),
        "n_estimators": Input(type="integer", default=568),
        "learning_rate": Input(type="number", default=0.05),
        "num_leaves": Input(type="integer", default=31)
    },
    outputs={
        "output_model": Output(type=AssetTypes.URI_FOLDER),
        "output_metrics": Output(type=AssetTypes.URI_FOLDER)
    },
    code="../src/pipeline",
    command="python train.py --input_data ${{inputs.input_data}} --output_model ${{outputs.output_model}} --output_metrics ${{outputs.output_metrics}} --n_estimators ${{inputs.n_estimators}} --learning_rate ${{inputs.learning_rate}} --num_leaves ${{inputs.num_leaves}}",
environment=f"credit-risk-pipeline-env-v5@latest",
    compute=compute_name
)

# Paso 3 — Evaluar modelo
evaluate_component = command(
    name="evaluate",
    display_name="Evaluar modelo",
    description="Verificar si el modelo nuevo supera el umbral de calidad AUC=0.75",
    inputs={
        "input_metrics": Input(type=AssetTypes.URI_FOLDER),
        "auc_threshold": Input(type="number", default=0.75)
    },
    outputs={
        "output_evaluation": Output(type=AssetTypes.URI_FOLDER)
    },
    code="../src/pipeline",
    command="python evaluate.py --input_metrics ${{inputs.input_metrics}} --output_evaluation ${{outputs.output_evaluation}} --auc_threshold ${{inputs.auc_threshold}}",
environment=f"credit-risk-pipeline-env-v5@latest",
    compute=compute_name
)

# Paso 4 — Registrar modelo
register_component = command(
    name="register",
    display_name="Registrar modelo",
    description="Registrar nueva versión del modelo en Azure ML Model Registry",
    inputs={
        "input_model": Input(type=AssetTypes.URI_FOLDER),
        "input_evaluation": Input(type=AssetTypes.URI_FOLDER),
        "model_name": Input(type="string", default="lightgbm-credit-risk"),
        "subscription_id": Input(type="string"),
        "resource_group": Input(type="string"),
        "workspace_name": Input(type="string")
    },
    outputs={},
    code="../src/pipeline",
    command="python register.py --input_model ${{inputs.input_model}} --input_evaluation ${{inputs.input_evaluation}} --model_name ${{inputs.model_name}} --subscription_id ${{inputs.subscription_id}} --resource_group ${{inputs.resource_group}} --workspace_name ${{inputs.workspace_name}}",
environment=f"credit-risk-pipeline-env-v5@latest",
    compute=compute_name
)

print("✅ Componentes del pipeline definidos")

✅ Componentes del pipeline definidos


In [23]:
# ─────────────────────────────────────────────
# ENSAMBLAR Y EJECUTAR EL PIPELINE
# ─────────────────────────────────────────────

@pipeline(
    name="credit_risk_retraining_pipeline",
    description="Pipeline de reentrenamiento automático del modelo de scoring crediticio",
    display_name="Credit Risk Retraining Pipeline"
)
def credit_risk_pipeline(input_data):
    # @pipeline es un decorador que convierte esta función
    # en un pipeline de Azure ML — encadena los pasos
    # automáticamente basándose en las dependencias entre outputs e inputs

    # Paso 1 — Preparar datos
    prep = prep_data_component(
        input_data=input_data
    )

    # Paso 2 — Entrenar con output del paso 1
    train = train_component(
        input_data=prep.outputs.output_data
        # El output de prep se convierte en input de train
        # Azure ML detecta esta dependencia y los ejecuta en orden
    )

    # Paso 3 — Evaluar con métricas del paso 2
    evaluate = evaluate_component(
        input_metrics=train.outputs.output_metrics
    )

    # Paso 4 — Registrar con modelo y evaluación
    register = register_component(
        input_model=train.outputs.output_model,
        input_evaluation=evaluate.outputs.output_evaluation,
        subscription_id=subscription_id,
        resource_group=resource_group,
        workspace_name=workspace_name
    )

    return {
        "prepared_data": prep.outputs.output_data,
        "trained_model": train.outputs.output_model,
        "evaluation": evaluate.outputs.output_evaluation
    }

# ─────────────────────────────────────────────
# CONFIGURAR INPUT Y LANZAR
# ─────────────────────────────────────────────

pipeline_job = credit_risk_pipeline(
    input_data=Input(
        type=AssetTypes.URI_FOLDER,
        path="azureml://datastores/workspaceblobstore/paths/data/processed/",
        # Ruta en Azure Blob Storage donde están los datos
        # workspaceblobstore es el storage por defecto del workspace
        mode=InputOutputModes.RO_MOUNT
        # RO_MOUNT: monta la carpeta en modo solo lectura
        # más eficiente que descargar los datos completos
    )
)

pipeline_job.settings.default_compute = compute_name
pipeline_job.settings.default_datastore = "workspaceblobstore"

# Enviar el pipeline a Azure ML
pipeline_run = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="credit_risk_retraining"
    # Agrupa todos los runs de este pipeline bajo el mismo experimento
)

print(f"✅ Pipeline enviado a Azure ML")
print(f"Run ID: {pipeline_run.name}")
print(f"Estado: {pipeline_run.status}")
print(f"URL: {pipeline_run.studio_url}")

Uploading pipeline (0.01 MBs): 100%|###| 11640/11640 [00:01<00:00, 7797.88it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


✅ Pipeline enviado a Azure ML
Run ID: honest_malanga_m294397h1z
Estado: NotStarted
URL: https://ml.azure.com/runs/honest_malanga_m294397h1z?wsid=/subscriptions/7a434baa-d49b-4814-9250-b27163c8f390/resourcegroups/rg-credit-risk-ml/workspaces/ws-credit-risk-ml&tid=adcc6d74-c336-483f-b3e0-508ef38f11d8


In [9]:
# ─────────────────────────────────────────────
# ACTUALIZAR ENTORNO CON azureml-mlflow
# ─────────────────────────────────────────────

pipeline_env_v2 = Environment(
    name="credit-risk-pipeline-env",
    description="Entorno para pipeline de reentrenamiento — v2 con azureml-mlflow",
    conda_file="../src/endpoint/environment.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
)

ml_client.environments.create_or_update(pipeline_env_v2)
print("✅ Entorno actualizado con azureml-mlflow")

✅ Entorno actualizado con azureml-mlflow


In [21]:
# ─────────────────────────────────────────────
# FORZAR NUEVO ENTORNO CON NOMBRE DIFERENTE
# ─────────────────────────────────────────────

pipeline_env_v3 = Environment(
    name="credit-risk-pipeline-env-v5",
    # Nombre diferente fuerza Azure ML a construir
    # una imagen nueva sin usar el cache anterior
    description="Entorno v2 — sin mlflow tracking, con joblib",
    conda_file="../src/endpoint/environment.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
)

ml_client.environments.create_or_update(pipeline_env_v3)
print("✅ Entorno v2 registrado")

✅ Entorno v2 registrado
